# PROYECTO YOLO AFORO - MINERÍA DE DATOS (MDA-611)
**Autor:** Antonio
**Hito 4: Análisis y Visualización**

Este cuaderno contiene todo el código fuente del proyecto, estructurado paso a paso, desde la configuración inicial hasta la generación del Dashboard final.


## Punto 1: Configuración e Instalación del Entorno

### 1.1 Instalación de Dependencias
Instalamos las librerías necesarias para la detección, transformación y visualización.

In [ ]:
!pip install ultralytics opencv-python pandas matplotlib nbformat -q

### 1.2 Configuración Centralizada (`config.py`)
Se definen las rutas, el modelo a utilizar, los umbrales de confianza y el diccionario de traducción al español.

In [ ]:
"""
config.py
---------
Configuración centralizada del proyecto YOLO Aforo.
"""
from pathlib import Path

VIDEO_SOURCE = 0
MODEL_PATH = "yolov8n.pt"
CONFIDENCE_THRESHOLD = 0.3
SAMPLING_INTERVAL_SECONDS = 3
OUTPUT_DIR = Path("data")
CSV_COLUMNS = ["dia_semana", "fecha", "hora", "clase_objeto"]
SHOW_VIDEO_WINDOW = True

CLEAN_DATASET_PATH = OUTPUT_DIR / "dataset_limpio.csv"
CLASSES_TO_KEEP = {"person": "persona"}

# Nuevos rangos de jornada (Mañana, Tarde y Noche)
MORNING_RANGE = ("06:00:00", "11:59:59")
AFTERNOON_RANGE = ("12:00:00", "18:59:59")
NIGHT_RANGE = ("19:00:00", "05:59:59")

CLEAN_CSV_COLUMNS = ["fecha", "dia", "hora", "clase_objeto", "bloque_horario"]

CLASS_TRANSLATIONS = {
    "person": "persona",
    "dog": "perro",
    "cat": "gato",
    "car": "coche",
    "motorcycle": "moto",
    "bicycle": "bicicleta",
    "cell phone": "celular",
    "chair": "silla",
    "bottle": "botella",
    "cup": "taza",
    "laptop": "portatil",
    "book": "libro",
    "backpack": "mochila"
}


## Punto 2: Fase 1 - Detección de Objetos con YOLO

### 2.1 Lógica de Detección (`detector.py`)
Aquí encapsulamos YOLOv8. Destaca nuestra implementación para **traducir las clases al español** en tiempo real y el uso de tracking para mantener el ID de las personas.

In [ ]:
"""
detector.py
-----------
Encapsula la lógica de detección de objetos con YOLO.
Dibuja las cajas manualmente para mostrar los nombres en español.
"""
from dataclasses import dataclass
from typing import List
import cv2
from ultralytics import YOLO
import numpy as np
import config

@dataclass
class Detection:
    class_name: str
    confidence: float
    track_id: int = None

class ObjectDetector:
    def __init__(self, model_path: str = config.MODEL_PATH,
                 confidence_threshold: float = config.CONFIDENCE_THRESHOLD):
        self.model = YOLO(model_path)
        self.confidence_threshold = confidence_threshold

    def detect(self, frame: np.ndarray) -> List[Detection]:
        results = self.model.track(frame, persist=True, verbose=False)[0]
        detections = []
        
        if results.boxes is None:
            return detections

        for box in results.boxes:
            confidence = float(box.conf[0])
            if confidence < self.confidence_threshold:
                continue
                
            class_id = int(box.cls[0])
            class_name_en = self.model.names[class_id]
            class_name_es = config.CLASS_TRANSLATIONS.get(class_name_en, class_name_en)
            track_id = int(box.id[0]) if box.id is not None else None
            
            detections.append(Detection(
                class_name=class_name_es, 
                confidence=confidence,
                track_id=track_id
            ))
        return detections

    def annotate(self, frame: np.ndarray):
        # En lugar de results.plot(), dibujamos manualmente para traducir
        results = self.model.track(frame, persist=True, verbose=False)[0]
        annotated_frame = frame.copy()
        
        if results.boxes is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = float(box.conf[0])
                
                if confidence < self.confidence_threshold:
                    continue
                    
                class_id = int(box.cls[0])
                class_name_en = self.model.names[class_id]
                class_name_es = config.CLASS_TRANSLATIONS.get(class_name_en, class_name_en)
                
                # Dibujar caja verde
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Escribir el texto en español
                label = f"{class_name_es} {confidence:.2f}"
                cv2.putText(annotated_frame, label, (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                            
        return annotated_frame


## Punto 3: Fase 2 - Colecta de Datos (Ingesta)

### 3.1 Lógica de Guardado Diario (`data_logger.py`)
Nuestro backend crea un archivo distinto por cada día (`dataset_crudo_Dia_Fecha.csv`) y guarda la hora exacta sin segundos para evitar ruido.

In [ ]:
"""
data_logger.py
--------------
Maneja la escritura del dataset crudo a disco (CSV).
Genera un archivo dinámico por cada día de la semana.
"""
import csv
from datetime import datetime
from pathlib import Path
from typing import List
import config
from detector import Detection

DIAS_SEMANA = {0: "Lunes", 1: "Martes", 2: "Miercoles", 3: "Jueves", 4: "Viernes", 5: "Sabado", 6: "Domingo"}

class CSVDataLogger:
    def __init__(self):
        config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    def _get_daily_path(self, timestamp: datetime) -> Path:
        dia_str = DIAS_SEMANA[timestamp.weekday()]
        fecha_str = timestamp.strftime("%Y-%m-%d")
        filename = f"dataset_crudo_{dia_str}_{fecha_str}.csv"
        return config.OUTPUT_DIR / filename

    def _ensure_file_exists(self, path: Path) -> None:
        if not path.exists():
            with open(path, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(config.CSV_COLUMNS)

    def log(self, detections: List[Detection], timestamp: datetime = None) -> int:
        timestamp = timestamp or datetime.now()
        dia_semana = DIAS_SEMANA[timestamp.weekday()]
        fecha = timestamp.strftime("%Y-%m-%d")
        hora_minuto = timestamp.strftime("%H:%M")
        daily_path = self._get_daily_path(timestamp)
        self._ensure_file_exists(daily_path)
        with open(daily_path, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            for det in detections:
                writer.writerow([dia_semana, fecha, hora_minuto, det.class_name])
        return len(detections)


### 3.2 Ejecución de la Colecta (`main.py`)
Este script abre la cámara, detecta personas, y guarda los registros. 
*(Nota: Ejecutar esta celda abrirá la ventana de la cámara. Presiona 'q' en el video para detenerlo).*

In [ ]:
"""
main.py
-------
Punto de entrada del sistema YOLO Aforo.
"""
import time
from datetime import datetime
import cv2
import config
from detector import ObjectDetector
from data_logger import CSVDataLogger

def run() -> None:
    print(f"[INFO] Cargando modelo {config.MODEL_PATH}...")
    detector = ObjectDetector()
    logger = CSVDataLogger()

    print(f"[INFO] Abriendo fuente de video: {config.VIDEO_SOURCE}")
    cap = cv2.VideoCapture(config.VIDEO_SOURCE)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir la fuente de video '{config.VIDEO_SOURCE}'.")

    print(f"[INFO] Muestreando cada {config.SAMPLING_INTERVAL_SECONDS}s. Presiona 'q' para detener.")

    last_sample_time = 0.0
    total_registros = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            now = time.time()

            if now - last_sample_time >= config.SAMPLING_INTERVAL_SECONDS:
                detections = detector.detect(frame)
                
                # Extraemos los nombres de TODO lo que ve YOLO
                nombres_objetos = [d.class_name for d in detections]
                # Filtramos solo las personas para el aforo
                personas = [d for d in detections if d.class_name == "persona"]

                timestamp = datetime.now()
                filas_escritas = logger.log(personas, timestamp=timestamp)
                total_registros += filas_escritas

                # Mostramos en consola todo lo detectado
                print(f"[{timestamp.strftime('%H:%M:%S')}] YOLO ve: {nombres_objetos} | Aforo guardado: {len(personas)} personas")

                last_sample_time = now

            if config.SHOW_VIDEO_WINDOW:
                annotated_frame = detector.annotate(frame)
                annotated_frame = cv2.resize(annotated_frame, (640, 480))
                cv2.imshow("YOLO Aforo - Fase 1", annotated_frame)
                
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print(f"[INFO] Finalizado. Registros de personas escritos: {total_registros}")

if __name__ == "__main__":
    run()


### 3.3 Ejecutando la Colecta
Corremos el script principal para capturar datos (se abrirá la cámara).

In [ ]:
# Descomenta la siguiente línea para probar la cámara en vivo
# %run main.py

import glob
archivos = glob.glob('data/dataset_crudo_*.csv')
print(f"Archivos crudos encontrados hasta ahora: {len(archivos)}")
for a in archivos:
    print(f" - {a}")

## Punto 4: Fase 4 - Transformación y Limpieza de Datos (DATA QUALITY)

### 4.1 Lógica de Limpieza (`transformer.py`)
Este es el núcleo de la ingeniería de datos. Lee todos los CSVs diarios, filtra falsos positivos (perros, celulares), **elimina duplicados exactos** para no inflar el aforo, y genera un reporte de calidad.

In [ ]:
"""
transformer.py
--------------
Transforma los datos crudos en un dataset limpio.
Clasifica en bloques: Mañana, Tarde y Noche.
"""
import glob
import pandas as pd
from datetime import datetime
import config

def load_raw_data() -> pd.DataFrame:
    print("[FASE 4] Cargando todos los archivos diarios...")
    archivos = glob.glob(str(config.OUTPUT_DIR / "dataset_crudo_*.csv"))
    if not archivos:
        raise FileNotFoundError("No se encontraron archivos dataset_crudo_*.csv")
    
    dfs = [pd.read_csv(f) for f in archivos]
    df = pd.concat(dfs, ignore_index=True)
    print(f" -> Se leyeron {len(archivos)} archivos. Total de registros crudos: {len(df)}")
    return df

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    print("[FASE 4] Iniciando limpieza de datos...")
    registros_iniciales = len(df)
    
    df = df.dropna(subset=['fecha', 'hora', 'clase_objeto'])
    nulos_eliminados = registros_iniciales - len(df)
    
    df = df[df['clase_objeto'] == 'person'].copy()
    df['clase_objeto'] = 'persona'
    no_personas_eliminadas = registros_iniciales - nulos_eliminados - len(df)
    
    df = df.drop_duplicates(subset=['fecha', 'hora', 'clase_objeto'])
    duplicados_eliminados = (registros_iniciales - nulos_eliminados - no_personas_eliminadas) - len(df)
    
    print(" --- REPORTE DE LIMPIEZA ---")
    print(f" -> Registros con nulos eliminados: {nulos_eliminados}")
    print(f" -> Registros no-persona eliminados: {no_personas_eliminadas}")
    print(f" -> Registros duplicados eliminados: {duplicados_eliminados}")
    print(f" -> Total de registros limpios conservados: {len(df)}")
    
    return df

def enrich_time_block(df: pd.DataFrame) -> pd.DataFrame:
    print("[FASE 4] Enriqueciendo datos (Mañana, Tarde, Noche)...")
    def obtener_bloque(hora_str):
        try:
            h = datetime.strptime(hora_str, "%H:%M").time()
            
            manana_inicio = datetime.strptime(config.MORNING_RANGE[0], "%H:%M:%S").time()
            manana_fin = datetime.strptime(config.MORNING_RANGE[1], "%H:%M:%S").time()
            
            tarde_inicio = datetime.strptime(config.AFTERNOON_RANGE[0], "%H:%M:%S").time()
            tarde_fin = datetime.strptime(config.AFTERNOON_RANGE[1], "%H:%M:%S").time()
            
            if manana_inicio <= h <= manana_fin:
                return "Mañana"
            elif tarde_inicio <= h <= tarde_fin:
                return "Tarde"
            else:
                return "Noche"
        except:
            return "Desconocido"
    
    df['bloque_horario'] = df['hora'].apply(obtener_bloque)
    return df

def transform() -> pd.DataFrame:
    df = load_raw_data()
    df = clean_data(df)
    df = enrich_time_block(df)
    
    if 'dia_semana' in df.columns:
        df = df.rename(columns={'dia_semana': 'dia'})
        
    columnas = ["fecha", "dia", "hora", "clase_objeto", "bloque_horario"]
    return df[columnas]

def save_clean_data(df: pd.DataFrame, path=config.CLEAN_DATASET_PATH) -> None:
    df.to_csv(path, index=False)
    print(f"[FASE 4] Dataset limpio guardado exitosamente en: {path}")


### 4.2 Script de Ejecución (`transform.py`)

In [ ]:
"""
transform.py
------------
Script de ejecución para la Fase 2.
Llama a las funciones del nuevo transformer.py que lee los archivos diarios.
"""
from transformer import transform, save_clean_data
import config

def run() -> None:
    print("[INFO] Iniciando Fase 2: Transformación de datos...")
    
    # Ejecuta la transformación (lee todos los dataset_crudo_*.csv)
    df_limpio = transform()
    
    # Guarda el resultado en dataset_limpio.csv
    save_clean_data(df_limpio, config.CLEAN_DATASET_PATH)
    
    print(f"[INFO] Transformación completada. Total de registros limpios: {len(df_limpio)}")

if __name__ == "__main__":
    run()


### 4.3 Ejecutando la Transformación y Mostrando Resultados
Corremos el pipeline y visualizamos el dataset limpio.

In [ ]:
%run transform.py

import pandas as pd
df_limpio = pd.read_csv('data/dataset_limpio.csv')
print("\n--- Muestra del Dataset Limpio ---")
display(df_limpio.head(10))

## Punto 5: Fase 6 - Visualización de Resultados y Dashboard

### 5.1 Lógica del Dashboard (`dashboard.py`)
Creamos un dashboard unificado que muestra el aforo total por día (barras) y la tendencia acumulada por hora (escalones).

In [ ]:
"""
dashboard.py
------------
Dashboard de Aforo - Análisis Integral (2x2).
Soluciona la superposición de días usando desplazamiento horizontal (Jitter).
"""
import pandas as pd
import matplotlib.pyplot as plt
import config

def generar_dashboard():
    print("[INFO] Leyendo dataset limpio...")
    df = pd.read_csv(config.CLEAN_DATASET_PATH)
    
    if df.empty:
        print("[WARN] El dataset está vacío.")
        return

    # Preparar datos generales
    orden_dias = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
    orden_bloques = ['Mañana', 'Tarde', 'Noche']
    df['hora_hh'] = df['hora'].str.slice(0, 2) + ":00"

    # Configurar el lienzo 2x2
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Dashboard de Aforo - Análisis Integral', fontsize=18, fontweight='bold')
    fig.subplots_adjust(hspace=0.5, wspace=0.3)

    # ==========================================
    # GRÁFICO 1 (Arriba Izq): Barras por Día y Bloque
    # ==========================================
    ax1 = axes[0, 0]
    pivot_df = df.groupby(['dia', 'bloque_horario']).size().unstack(fill_value=0)
    for col in orden_bloques:
        if col not in pivot_df.columns: pivot_df[col] = 0
    pivot_df = pivot_df[orden_bloques].reindex(orden_dias, fill_value=0)
    
    pivot_df.plot(kind='bar', ax=ax1, color=['#FFD700', '#FF8C00', '#4A55A2'], edgecolor='black', width=0.7)
    ax1.set_title('Aforo por Día y Bloque Horario')
    ax1.set_xlabel('Día de la Semana')
    ax1.set_ylabel('Total Personas')
    ax1.set_xticklabels(orden_dias, rotation=45)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    ax1.legend(title='Bloque')
    for p in ax1.patches:
        if p.get_height() > 0:
            ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')

    # ==========================================
    # GRÁFICO 2 (Arriba Der): Pastel por Bloque Horario
    # ==========================================
    ax2 = axes[0, 1]
    conteo_bloques = df.groupby('bloque_horario').size().reindex(orden_bloques, fill_value=0)
    conteo_bloques = conteo_bloques[conteo_bloques > 0]
    
    if not conteo_bloques.empty:
        ax2.pie(conteo_bloques, labels=conteo_bloques.index, autopct='%1.1f%%', 
                colors=['#FFD700', '#FF8C00', '#4A55A2'], startangle=90, textprops={'fontsize': 12})
    ax2.set_title('Distribución Porcentual por Bloque')

    # ==========================================
    # GRÁFICO 3 (Abajo Izq): Barras por Hora
    # ==========================================
    ax3 = axes[1, 0]
    conteo_horas = df.groupby('hora_hh').size().reset_index(name='total')
    ax3.bar(conteo_horas['hora_hh'], conteo_horas['total'], color='#3498DB', edgecolor='black', alpha=0.8)
    ax3.set_title('Aforo Total por Hora del Día')
    ax3.set_xlabel('Hora (HH:00)')
    ax3.set_ylabel('Total Personas')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
    for p in ax3.patches:
        if p.get_height() > 0:
            ax3.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')

    # ==========================================
    # GRÁFICO 4 (Abajo Der): Línea Acumulada CON DESPLAZAMIENTO (JITTER)
    # ==========================================
    ax4 = axes[1, 1]
    colores_lineas = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    estilos_linea = ['-', '--', '-.', ':']
    marcadores = ['o', 's', '^', 'D']
    
    # Obtener todas las horas únicas ordenadas
    horas_unicas = sorted(df['hora'].unique())
    hora_a_idx = {h: i for i, h in enumerate(horas_unicas)} # Mapear hora a un índice numérico
    
    fechas_unicas = df['fecha'].unique()
    num_dias = len(fechas_unicas)
    
    for i, fecha in enumerate(fechas_unicas):
        df_dia = df[df['fecha'] == fecha].copy()
        df_dia['acumulado'] = range(1, len(df_dia) + 1)
        
        nombre_dia = df_dia['dia'].iloc[0]
        color = colores_lineas[i % len(colores_lineas)]
        estilo = estilos_linea[i % len(estilos_linea)]
        marcador = marcadores[i % len(marcadores)]
        
        # Calcular desplazamiento (Jitter): si hay 3 días, los offsets serán -0.15, 0, +0.15
        offset = (i - (num_dias - 1) / 2) * 0.15
        
        # Crear posiciones X desplazadas
        x_pos = [hora_a_idx[h] + offset for h in df_dia['hora']]
        
        # Graficar con estilos distintos y el eje X desplazado
        ax4.plot(x_pos, df_dia['acumulado'], marker=marcador, linestyle=estilo,
                 label=f"{nombre_dia}", color=color, linewidth=2.5, markersize=8, drawstyle='steps-post')
        
        # Rellenar debajo de la línea
        ax4.fill_between(x_pos, df_dia['acumulado'], alpha=0.05, color=color, step='post')
        
        # Poner los números sobre cada punto
        for x, y in zip(x_pos, df_dia['acumulado']):
            ax4.text(x, y + 0.15, str(y), ha='center', va='bottom', fontweight='bold', color=color)

    # Configurar el eje X para que muestre las horas en lugar de números
    ax4.set_xticks(range(len(horas_unicas)))
    ax4.set_xticklabels(horas_unicas, rotation=45)
    
    ax4.set_title('Tendencia Acumulada por Hora (con desplazamiento)')
    ax4.set_xlabel('Hora del día')
    ax4.set_ylabel('Personas Acumuladas')
    ax4.legend(loc='upper left')
    ax4.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    ruta_salida = 'data/dashboard_aforo.png'
    plt.savefig(ruta_salida, dpi=150)
    print(f"[INFO] Dashboard con Jitter guardado en: {ruta_salida}")

if __name__ == "__main__":
    generar_dashboard()


### 5.2 Ejecutando el Dashboard
Generamos la imagen final y la renderizamos en el cuaderno.

In [ ]:
%run dashboard.py

# Mostrar la imagen generada en el Notebook
from IPython.display import Image, display
try:
    display(Image(filename='data/dashboard_aforo.png'))
except FileNotFoundError:
    print("Aún no se ha generado el dashboard. Ejecuta la celda anterior.")